# Day 20 — Pandas Mini-Project: Data Cleaning and Validation Pipeline

This notebook covers End-to-End Data Cleaning Pipeline, String Normalization, Outlier Treatment, Imputation, and Validated Aggregations

## Validation goals
- Synthesize Week 3 concepts into a structured, reproducible data validation pipeline
- Perform string standardization (`.str.strip()`, `.str.lower()`, `.str.title()`) to normalize casing and whitespace anomalies[cite: 19]
- Execute selective missing data drop strategies (`dropna(subset=['txn_id'])`) and categorical/numeric imputations[cite: 19]
- Handle corrupt values (`-999.0`) and invalid strings cleanly using boolean masking and `pd.to_datetime(errors='coerce')`[cite: 19]
- Generate clean regional performance summaries via `.groupby()` aggregations and export cleaned artifacts[cite: 19]

To synthesize the concepts covered throughout Week 3—including data ingestion, structured indexing, conditional filtering, missing value imputation, and group-wise aggregations—you will build a comprehensive, zero-loop Data Cleaning and Validation Pipeline.

Real-world datasets contain structural anomalies, trailing whitespace, malformed timestamps, mixed-type numeric columns, and extreme outliers. This mini-project ingests a deliberately corrupted customer transaction dataset, sanitizes it, and outputs a validated, analysis-ready table.

### Project Architecture & Workflow

1. Ingestion & Schema Enforcement: Load raw data while handling corrupt delimiters and mixed types.

2. String & Text Normalization: Strip whitespace, standardize casing, and parse date strings.

3. Missing Value Treatment: Apply selective dropping for critical foreign keys and domain-specific imputation for numerical/categorical fields.

4. Outlier Removal: Filter statistical anomalies using vectorized boundaries.

5. Validation & Aggregation: Confirm schema integrity and output clean aggregated summaries.

In [1]:
import numpy as np
import pandas as pd

# ==========================================
# Phase 1: Simulate Messy Raw Data Ingestion
# ==========================================
np.random.seed(42)
n_rows = 1000

raw_data = {
    "txn_id": [f"txn_{i:04d}" if i % 50 != 0 else np.nan for i in range(n_rows)],
    "customer_email": [f" user_{i}@company.com " if i % 3 != 0 else "CORRUPT_EMAIL" for i in range(n_rows)],
    "store_region": np.random.choice([" north ", "south", "EAST", "west ", np.nan], size=n_rows, p=[0.25, 0.25, 0.25, 0.15, 0.1]),
    "transaction_amount": [np.random.uniform(10.0, 500.0) if i % 20 != 0 else -999.0 for i in range(n_rows)], # Outlier/corruption marker
    "timestamp": pd.date_range("2026-10-01", periods=n_rows, freq="h").astype(str),
}

# Inject manual missing timestamps
timestamps_list = list(raw_data["timestamp"])
for idx in [12, 45, 89, 204]:
    timestamps_list[idx] = np.nan
raw_data["timestamp"] = timestamps_list

df_raw = pd.DataFrame(raw_data)

print("--- Raw Dataset Sample (Before Cleaning) ---")
print(df_raw.head(3))
print(f"Initial Row Count: {len(df_raw)}\n")


# ==========================================
# Phase 2: Structural Cleaning & Normalization
# ==========================================
def clean_transaction_data(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()

    # 1. Drop rows with missing critical identifiers (txn_id)
    df_clean = df_clean.dropna(subset=["txn_id"])

    # 2. String Normalization: Strip whitespace, lower case, fix casing
    df_clean["customer_email"] = df_clean["customer_email"].str.strip().str.lower()
    df_clean["store_region"] = df_clean["store_region"].str.strip().str.title()

    # Mask invalid emails containing 'corrupt'
    df_clean.loc[df_clean["customer_email"].str.contains("corrupt", na=False), "customer_email"] = np.nan

    # 3. Parse Datetimes
    df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"], errors="coerce")

    # 4. Handle Numerical Outliers & Errors (-999.0 markers)
    df_clean.loc[df_clean["transaction_amount"] < 0.0, "transaction_amount"] = np.nan

    # 5. Impute Missing Numerical Values with Group-wise Median
    # Fallback to global median if group is entirely NaN
    df_clean["transaction_amount"] = df_clean["transaction_amount"].fillna(
        df_clean["transaction_amount"].median()
    )

    # 6. Impute Categorical Region with Mode / Constant
    df_clean["store_region"] = df_clean["store_region"].fillna("Unknown")

    # 7. Drop duplicate transactions if any exist based on txn_id
    df_clean = df_clean.drop_duplicates(subset=["txn_id"], keep="first")

    return df_clean

df_processed = clean_transaction_data(df_raw)

print("--- Cleaned Dataset Summary ---")
print(df_processed.info())
print("\nMissing Values Post-Cleaning:")
print(df_processed.isna().sum())


# ==========================================
# Phase 3: Validated Aggregations & Reporting
# ==========================================
regional_performance = df_processed.groupby("store_region", as_index=False).agg(
    total_transactions=("txn_id", "count"),
    total_revenue=("transaction_amount", "sum"),
    average_ticket_size=("transaction_amount", "mean")
)

print("\n--- Final Aggregated Regional Report ---")
print(regional_performance)

# Save cleaned output artifact
df_processed.to_csv("cleaned_transactions.csv", index=False)

--- Raw Dataset Sample (Before Cleaning) ---
     txn_id        customer_email store_region  transaction_amount  \
0       NaN         CORRUPT_EMAIL        south         -999.000000   
1  txn_0001   user_1@company.com           nan          100.715135   
2  txn_0002   user_2@company.com          EAST          275.531464   

             timestamp  
0  2026-10-01 00:00:00  
1  2026-10-01 01:00:00  
2  2026-10-01 02:00:00  
Initial Row Count: 1000

--- Cleaned Dataset Summary ---
<class 'pandas.DataFrame'>
Index: 980 entries, 1 to 999
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   txn_id              980 non-null    str           
 1   customer_email      653 non-null    str           
 2   store_region        980 non-null    str           
 3   transaction_amount  980 non-null    float64       
 4   timestamp           976 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(1)